In [ ]:
import ee
import geemap
from utils.variables import PROJECT, ANALYSIS_END_YR, GLC_LABELS, DRIVER_LABELS

from absolute_effectiveness_functions import (
    get_test_sites,
    set_start_yr,
    get_site_geom,
    check_start_yr,
    process_GLC,
    process_GPW,
    process_NFW,
    process_HGFC,
    get_habitat_raster,
    calc_habitat_extent_score,
    get_edge_distance_raster,
    calc_edge_distance_score,
    get_patch_size_raster,
    calc_patch_size_score,
    calc_intactness_score,
    calc_habitat_condition_score,
    get_habitat_loss_raster,
    calc_habitat_loss_score,
    get_driver_class_image,
    calc_class_area_and_pct,
    get_habitat_class_image,
    get_s2_med_composite,
    translate_results,
)

ee.Authenticate()
ee.Initialize(project=PROJECT)

In [ ]:
# Select site by WDPAID (one from the list of 30 in variables.py)
site_id = 352159

# Get test sites and site-specific information
test_sites = get_test_sites()
START_YR = set_start_yr(test_sites, site_id)
site_geom = get_site_geom(test_sites, site_id)

# Check if PA designation year is valid for analysis
check_start_yr(START_YR)

In [ ]:
# Process all input datasets
GLC_processed = process_GLC(test_sites, START_YR)
GPW_processed = process_GPW(START_YR)
NFW_processed = process_NFW(test_sites)
HGFC_processed = process_HGFC(START_YR)

# Calculate Habitat Extent score
habitat_raster = get_habitat_raster(
    GLC_processed, HGFC_processed, GPW_processed, NFW_processed
)
habitat_extent_score = calc_habitat_extent_score(habitat_raster, site_geom)

# Calculate Habitat Intactness score
edge_distance_raster = get_edge_distance_raster(habitat_raster, site_geom)
edge_distance_score = calc_edge_distance_score(edge_distance_raster, site_geom)
patch_size_raster = get_patch_size_raster(habitat_raster, site_geom)
patch_size_score = calc_patch_size_score(patch_size_raster, site_geom)
habitat_intactness_score = calc_intactness_score(edge_distance_score, patch_size_score)

# Calculate overall Habitat Condition score
habitat_condition_score = calc_habitat_condition_score(
    habitat_extent_score, habitat_intactness_score
)

# Calculate Habitat Loss score
habitat_loss_raster, habitat_start_raster = get_habitat_loss_raster(
    GLC_processed, GPW_processed, HGFC_processed, START_YR
)
habitat_loss_score = calc_habitat_loss_score(
    habitat_loss_raster, habitat_start_raster, site_geom
)

# Analyze drivers and types of habitat loss
driver_class = get_driver_class_image(GLC_processed, GPW_processed, habitat_loss_raster)
driver_dict = calc_class_area_and_pct(driver_class, site_geom)
habitat_class = get_habitat_class_image(GLC_processed, habitat_loss_raster, START_YR)
habitat_dict = calc_class_area_and_pct(habitat_class, site_geom)

# Create Sentinel-2 composites (for visualization purposes only)
start_s2_composite = get_s2_med_composite(site_geom, START_YR)
end_s2_composite = get_s2_med_composite(site_geom, ANALYSIS_END_YR)

# Print all results
print(f"Site ID: {site_id}")
print(f"Analysis Period: {START_YR} - {ANALYSIS_END_YR}")
print(f"\nHabitat Extent Score: {habitat_extent_score:.4f}")
print(f"Edge Distance Score: {edge_distance_score:.4f}")
print(f"Patch Size Score: {patch_size_score:.4f}")
print(f"Habitat Intactness Score: {habitat_intactness_score:.4f}")
print(f"Habitat Condition Score: {habitat_condition_score:.4f}")
print(f"Habitat Loss Score: {habitat_loss_score:.4f}")
print("\nDrivers of habitat loss:")
translate_results(driver_dict, DRIVER_LABELS)
print("\nTop 4 types of habitat lost:")
translate_results(habitat_dict, GLC_LABELS)

In [ ]:
# ============================================================================
# VISUALIZATION
# ============================================================================

# from utils.variables import (
#     GLC_PALETTE,
#     MAX_EDGE_DIST,
#     MAX_PATCH_SIZE
# )
# from math import sqrt

# Sentinel-2 RGB viz parameters
s2_rgb_viz = {
    "min": 0.0,
    "max": 0.3,
    "bands": ["B4", "B3", "B2"],
}

Map = geemap.Map()
Map.add_basemap("Esri.WorldImagery")

# Map.addLayer(start_s2_composite, s2_rgb_viz, "S2, Start Year", 0)
# Map.addLayer(end_s2_composite, s2_rgb_viz, "S2, End Year", 0)
# Map.addLayer(GLC_processed.select("GLC_2022"), {"min": 0, "max": 36, "palette": GLC_PALETTE}, "Global Land Cover", 0)
# Map.addLayer(GPW_processed.select("GPW_2022").selfMask(), {"min": 1, "max": 2, "palette": ["#ffcd73", "#ff9916"]}, "Global Pasture Watch", 0)
# Map.addLayer(NFW_processed.selfMask(), {"palette": "teal"}, "Natural Forests of the World", 0)
# Map.addLayer(HGFC_processed.selfMask(), {"min": START_YR - 2000, "max": ANALYSIS_END_YR - 2000, "palette": ["yellow", "red"]}, "Hansen Global Forest Change", 0)
# Map.addLayer(habitat_raster.clip(site_geom), {"palette": GLC_PALETTE}, "Habitat extent")
# Map.addLayer(edge_distance_raster, {"min": 0, "max": MAX_EDGE_DIST, "palette": ["red", "white"]}, "Edge distance", 0)
# Map.addLayer(patch_size_raster.reproject(crs=habitat_raster.projection(), scale=sqrt((MAX_PATCH_SIZE * 1000000) / 1024)), {'min':1, 'max':1024, 'palette': ['red', 'white']}, 'Patch size')
# Map.addLayer(habitat_start_raster.selfMask().clip(site_geom), {'palette': GLC_PALETTE}, "Habitat at start year")
# Map.addLayer(habitat_loss_raster.selfMask().clip(site_geom), {'palette': ['orange']}, "Habitat loss (binary)")
# Map.addLayer(driver_class.selfMask(), {"min": 1, "max": 4, "palette": ["yellow", "red", "orange", "green"]}, "Drivers of habitat loss")
# Map.addLayer(habitat_class.selfMask(), {"palette": GLC_PALETTE}, "Types of habitat lost")
# Map.addLayer(site_geom, {"color": "yellow"}, "Test site")

Map.centerObject(site_geom)

Map